In [1]:
import pandas as pd

## Step 1: Processing exchange rate dates and aggregating monthy

In [2]:
df_exc = pd.read_csv("data/usd_tzs_2022_2024.csv")
df_exc.head()

,Date,Price,Open,High,Low,Vol.,Change %
0,12/31/2024,"2,425.00","2,435.00","2,435.00","2,405.00",NaN,-0.82%
1,12/30/2024,"2,445.00","2,415.00","2,470.00","2,415.00",NaN,0.82%
2,12/27/2024,"2,425.00","2,420.00","2,432.50","2,407.50",NaN,1.46%
3,12/26/2024,"2,390.00","2,390.00","2,390.00","2,390.00",NaN,0.00%
4,12/25/2024,"2,390.00","2,390.00","2,390.00","2,390.00",NaN,-1.24%


In [3]:
df_exc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 781 entries, 0 to 780
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Date      781 non-null    object
 1   Price     781 non-null    object
 2   Open      781 non-null    object
 3   High      781 non-null    object
 4   Low       781 non-null    object
 5   Vol.      266 non-null    object
 6   Change %  781 non-null    object
dtypes: object(7)
memory usage: 42.8+ KB


In [4]:
# Strip commas, spaces, and ensure numeric
df_exc["Price"] = (
    df_exc["Price"]
    .astype(str)                # force to string
    .str.replace(",", "", regex=False)  # remove commas
    .str.strip()                # remove spaces
)

df_exc["Price"] = pd.to_numeric(df_exc["Price"], errors="coerce")


In [5]:
# 1. Convert Date column to datetime
df_exc["Date"] = pd.to_datetime(df_exc["Date"], format="%m/%d/%Y")

# 2. Keep only Date and Price, ensure Price is numeric
df_exc["Price"] = pd.to_numeric(df_exc["Price"], errors="coerce")
df_exc = df_exc[["Date", "Price"]]

# 3. Resample by month end and take mean of Price
df_exc_monthly = (
    df_exc.set_index("Date")
    .resample("ME")["Price"]
    .mean()
    .reset_index()
)

# 4. Add YearMonth column
df_exc_monthly["YearMonth"] = df_exc_monthly["Date"].dt.strftime("%Y-%m")

# 5. Compute month-to-month percentage change in rate
df_exc_monthly["Rate_Change_%"] = (
    df_exc_monthly["Price"].pct_change(fill_method=None) * 100
)

# 6. Drop Date if you only want YearMonth
df_exc_monthly = df_exc_monthly[["YearMonth", "Price", "Rate_Change_%"]].rename(
    columns={"Price": "USD/TZS_Rate"}
)
df_exc_monthly["Rate_Change_%"] = df_exc_monthly["Rate_Change_%"].fillna(0)
df_exc_monthly.head()

,YearMonth,USD/TZS_Rate,Rate_Change_%
0,2022-01,2303.380952,0.000000
1,2022-02,2308.800000,0.235265
2,2022-03,2312.565217,0.163081
3,2022-04,2317.761905,0.224715
4,2022-05,2320.818182,0.131863


## Step 2: Processing CPI and aggregating monthy

In [6]:
df_cpi = pd.read_csv("data/tanzania_cpi_2022_2024.csv")
df_cpi.head()

,date,inflation_rate_pct,all_items_index
0,2022-01-01,3.997044,105.591638
1,2022-02-01,3.673581,106.201764
2,2022-03-01,3.552367,107.085012
3,2022-04-01,3.783246,107.877973
4,2022-05-01,4.026812,108.420516


In [7]:
df_cpi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   date                35 non-null     object 
 1   inflation_rate_pct  35 non-null     float64
 2   all_items_index     35 non-null     float64
dtypes: float64(2), object(1)
memory usage: 972.0+ bytes


In [8]:
df_cpi["date"] = pd.to_datetime(df_cpi["date"], format="%Y-%m-%d")
df_cpi_monthly = df_cpi.copy()
df_cpi_monthly["YearMonth"] = df_cpi_monthly["date"].dt.strftime("%Y-%m")
df_cpi_monthly = df_cpi_monthly.drop(columns=["date","all_items_index"])

df_cpi_monthly = df_cpi_monthly.rename(columns={"inflation_rate_pct": "inflation_rate_%", "inflation_rate_pct": "inflation_rate_%"})

df_cpi_monthly.head()

,inflation_rate_%,YearMonth
0,3.997044,2022-01
1,3.673581,2022-02
2,3.552367,2022-03
3,3.783246,2022-04
4,4.026812,2022-05


## Step 3: Processing Headlines and aggregating monthy

In [9]:
df_hdl = pd.read_csv("data/tz_headlines_labelled.csv")
df_hdl.head()

,date,headline,url,source,scraped_on,year_month,relevant,category,sentiment,reason,sentition
0,2024-01-01,Economists’ advice to government as Tanzania w...,https://www.thecitizen.co.tz/tanzania/news/nat...,The Citizen,2026-05-20,2024-01,False,NaN,NaN,pure politics,NaN
1,2024-01-01,Tanzania moves to bring down soaring sugar prices,https://www.thecitizen.co.tz/tanzania/news/nat...,The Citizen,2026-05-20,2024-01,True,Agriculture,Neutral,sugar prices announcement,NaN
2,2023-12-31,Disagreement on days set to climb Mt. Kilimanjaro,https://www.thecitizen.co.tz/tanzania/news/nat...,The Citizen,2026-05-20,2023-12,False,NaN,NaN,pure sports,NaN
3,2023-12-31,Why government’s directive on boarding schools...,https://www.thecitizen.co.tz/tanzania/news/nat...,The Citizen,2026-05-20,2023-12,False,NaN,NaN,pure politics,NaN
4,2023-12-30,TRC receives new locomotives and passenger cars,https://www.thecitizen.co.tz/tanzania/news/nat...,The Citizen,2026-05-20,2023-12,True,Transport,Neutral,new locomotives and cars,NaN


In [10]:
print(len(df_hdl), ": Size of data set prior to droppping irrelvant headlines")
df_hdl = df_hdl[df_hdl["relevant"] == True]
print(len(df_hdl), ": Size of data set after droppping irrelvant headlines")

5684 : Size of data set prior to droppping irrelvant headlines
3865 : Size of data set after droppping irrelvant headlines


In [11]:
# Droppping unnecessary  columns
df_hdl_clean = df_hdl[["date", "headline", "category", "sentiment"]]
df_hdl_clean.head()

,date,headline,category,sentiment
1,2024-01-01,Tanzania moves to bring down soaring sugar prices,Agriculture,Neutral
4,2023-12-30,TRC receives new locomotives and passenger cars,Transport,Neutral
5,2023-12-30,Tanzania forecasts Sh22.5 trillion PPP boom by...,Investment,Positive
8,2023-12-28,Tanzanians count down the days before SGR serv...,Transport,Neutral
9,2023-12-27,Why cement prices are high despite surplus pro...,General,Negative


In [12]:
df_hdl_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3865 entries, 1 to 5683
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   date       3865 non-null   object
 1   headline   3865 non-null   object
 2   category   3865 non-null   object
 3   sentiment  3863 non-null   object
dtypes: object(4)
memory usage: 151.0+ KB


In [13]:
df_hdl_clean = df_hdl_clean.copy()

df_hdl_clean["date"] = pd.to_datetime(df_hdl_clean["date"], format="%Y-%m-%d", errors="coerce")
df_hdl_clean["YearMonth"] = df_hdl_clean["date"].dt.strftime("%Y-%m")

df_hdl_clean["category"] = df_hdl_clean["category"].astype("category")
df_hdl_clean["sentiment"] = df_hdl_clean["sentiment"].astype("category")

df_hdl_clean = df_hdl_clean.drop(columns=["headline","date"])

df_hdl_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3865 entries, 1 to 5683
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   category   3865 non-null   category
 1   sentiment  3863 non-null   category
 2   YearMonth  3865 non-null   object  
dtypes: category(2), object(1)
memory usage: 68.4+ KB


In [14]:
df_hdl_clean.head()

,category,sentiment,YearMonth
1,Agriculture,Neutral,2024-01
4,Transport,Neutral,2023-12
5,Investment,Positive,2023-12
8,Transport,Neutral,2023-12
9,General,Negative,2023-12


In [15]:
# 1. Compute total headlines per month
headline_counts = (
    df_hdl_clean.groupby("YearMonth", observed=True)
    .size()
    .reset_index(name="num_headlines")
)

# 2. Top category per month (percentage only, no raw count)
top_category = (
    df_hdl_clean.groupby(["YearMonth", "category"], observed=True)
    .size()
    .reset_index(name="count")
)

# Compute percentage within each month
top_category["category %"] = (
    top_category.groupby("YearMonth")["count"]
    .transform(lambda x: x / x.sum() * 100)
).round(2)

# Pick the top category by percentage
top_category = (
    top_category.sort_values(["YearMonth", "category %"], ascending=[True, False])
    .groupby("YearMonth")
    .first()
    .reset_index()
)

# Drop the raw count column
top_category = top_category.drop(columns=["count"])

# 3. Sentiment distribution per month (percentages)
sentiment_counts = (
    df_hdl_clean.groupby(["YearMonth", "sentiment"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

sentiment_cols = [c for c in sentiment_counts.columns if c != "YearMonth"]
sentiment_counts[sentiment_cols] = (
    sentiment_counts[sentiment_cols].div(sentiment_counts[sentiment_cols].sum(axis=1), axis=0) * 100
).round(2)

sentiment_counts["Avg_sentiment"] = sentiment_counts[sentiment_cols].idxmax(axis=1)

# 4. Merge everything: headline totals + top category + sentiment
df_hdl_monthly = (
    headline_counts
    .merge(top_category, on="YearMonth")
    .merge(sentiment_counts, on="YearMonth")
)

df_hdl_monthly.head()


,YearMonth,num_headlines,category,category %,Negative,Neutral,Positive,Avg_sentiment
0,2022-01,43,General,37.21,16.28,62.79,20.93,Neutral
1,2022-02,33,General,24.24,9.09,60.61,30.30,Neutral
2,2022-03,45,General,35.56,8.89,71.11,20.00,Neutral
3,2022-04,41,General,24.39,17.07,46.34,36.59,Neutral
4,2022-05,44,General,22.73,15.91,40.91,43.18,Positive


## Step 3: Consolidatig aggreagted data

In [16]:
df_hdl_monthly.head()

,YearMonth,num_headlines,category,category %,Negative,Neutral,Positive,Avg_sentiment
0,2022-01,43,General,37.21,16.28,62.79,20.93,Neutral
1,2022-02,33,General,24.24,9.09,60.61,30.30,Neutral
2,2022-03,45,General,35.56,8.89,71.11,20.00,Neutral
3,2022-04,41,General,24.39,17.07,46.34,36.59,Neutral
4,2022-05,44,General,22.73,15.91,40.91,43.18,Positive


In [17]:
df_cpi_monthly.head()

,inflation_rate_%,YearMonth
0,3.997044,2022-01
1,3.673581,2022-02
2,3.552367,2022-03
3,3.783246,2022-04
4,4.026812,2022-05


In [18]:
df_exc_monthly.head()

,YearMonth,USD/TZS_Rate,Rate_Change_%
0,2022-01,2303.380952,0.000000
1,2022-02,2308.800000,0.235265
2,2022-03,2312.565217,0.163081
3,2022-04,2317.761905,0.224715
4,2022-05,2320.818182,0.131863


In [19]:
df_merge = pd.merge(
    df_hdl_monthly, 
    df_cpi_monthly, 
    on="YearMonth", 
    how="inner"
)

# Merge with exchange rates
df_merge = pd.merge(
    df_merge, 
    df_exc_monthly, 
    on="YearMonth", 
    how="inner"
)

df_merge = df_merge.rename(
    columns={
        "category": "Top_Category",
        "count": "Headline_Count",
        "Negative": "Negative %",
        "Neutral": "Neutral %",
        "Positive": "Positive %",
        "Avg_sentiment": "Dominant_Sentiment",
        "inflation_rate_%": "Inflation %",
        "all_items_index": "CPI_Index"
    }
)

# Final consolidated DataFrame
df_merge.head()


,YearMonth,num_headlines,Top_Category,category %,Negative %,Neutral %,Positive %,Dominant_Sentiment,Inflation %,USD/TZS_Rate,Rate_Change_%
0,2022-01,43,General,37.21,16.28,62.79,20.93,Neutral,3.997044,2303.380952,0.000000
1,2022-02,33,General,24.24,9.09,60.61,30.30,Neutral,3.673581,2308.800000,0.235265
2,2022-03,45,General,35.56,8.89,71.11,20.00,Neutral,3.552367,2312.565217,0.163081
3,2022-04,41,General,24.39,17.07,46.34,36.59,Neutral,3.783246,2317.761905,0.224715
4,2022-05,44,General,22.73,15.91,40.91,43.18,Positive,4.026812,2320.818182,0.131863


In [20]:
df_merge

,YearMonth,num_headlines,Top_Category,category %,Negative %,Neutral %,Positive %,Dominant_Sentiment,Inflation %,USD/TZS_Rate,Rate_Change_%
0,2022-01,43,General,37.21,16.28,62.79,20.93,Neutral,3.997044,2303.380952,0.000000
1,2022-02,33,General,24.24,9.09,60.61,30.30,Neutral,3.673581,2308.800000,0.235265
2,2022-03,45,General,35.56,8.89,71.11,20.00,Neutral,3.552367,2312.565217,0.163081
3,2022-04,41,General,24.39,17.07,46.34,36.59,Neutral,3.783246,2317.761905,0.224715
4,2022-05,44,General,22.73,15.91,40.91,43.18,Positive,4.026812,2320.818182,0.131863
5,2022-06,60,Banking,20.00,8.33,70.00,21.67,Neutral,4.436698,2326.090909,0.227193
6,2022-07,56,General,32.14,3.57,78.57,17.86,Neutral,4.537499,2327.000000,0.039082
7,2022-08,96,General,28.12,6.25,73.96,19.79,Neutral,4.649161,2326.695652,-0.013079
8,2022-09,176,General,21.59,7.95,69.89,22.16,Neutral,4.839360,2326.318182,-0.016223
9,2022-10,186,Agriculture,24.19,8.60,71.51,19.89,Neutral,4.942250,2326.523810,0.008839


In [21]:
df_merge.to_csv("data/Visualization_Data.csv", index=False)